# Phase 2: Real Concept Generation Sample + Dual Generation (Unseeded / Seeded)

Covers plan sections **2.1** (draw the generation sample) and **2.2** (generate concepts, unseeded and seeded). Section 2.3 (manual review) is intentionally not pre-built here -- it's iterative human judgment (read examples, rewrite criteria, rescore), not a run-to-completion script. The review-sheet-printer and Jaccard-merge-checker from your plan are included as ready-to-run cells for when you get there.

**Open question before you run 2.3**: how do you want to reconcile the unseeded and seeded concept lists into one final 12-16? Worth deciding before you're looking at two lists.


## 1. Setup

In [ ]:
%pip install -q text_lloom openai fastembed pandas

In [ ]:
import os
import pickle
import json
import itertools
import pandas as pd

pd.set_option('display.max_colwidth', 200)

import text_lloom.workbench as wb
from text_lloom.llm import Model, EmbedModel

In [ ]:
%pip install --upgrade certifi -q

import certifi, os
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['SSL_CERT_DIR'] = os.path.dirname(certifi.where())

import nltk
nltk.download('punkt_tab')

## 2. Load comments + bat_score, merge

`master_comments_filtered.csv` has no `bat_score` column -- it lives in the posts-level `bat_score_pos.csv`. Merging it in via `post_id` before anything else, since 2.1.2's severity band needs it.

In [ ]:
OUTPUT_DIR = "/Users/nadia/Desktop/redditRun_june/comment_data/"

assert os.path.isdir(OUTPUT_DIR), f"OUTPUT_DIR not found: {OUTPUT_DIR}"

c = pd.read_csv(os.path.join(OUTPUT_DIR, 'master_comments_filtered.csv'))
bat = pd.read_csv(os.path.join(OUTPUT_DIR, 'bat_score_pos.csv'), usecols=['post_id', 'bat_score'])

print(f"Comments: {len(c):,} rows, columns: {c.columns.tolist()}")
print(f"BAT scores: {len(bat):,} rows")

c['post_id'] = c['post_id'].astype(str)
bat['post_id'] = bat['post_id'].astype(str)
c = c.merge(bat, on='post_id', how='left')

n_missing = c['bat_score'].isna().sum()
print(f"\nAfter merge: {n_missing:,} comments have no matching bat_score "
      f"(expected 0, since comments were already filtered to bat_score>0 target posts)")
if n_missing:
    print("Some comments didn't match a bat_score -- check post_id types/values before proceeding.")

TEXT_COL = "body"
ID_COL = "id"                      # was "comment_id" in the sketch -- not a real column
SUBREDDIT_COL = "subreddit_source"  # was "subreddit" in the sketch -- not a real column

print(f"\nbat_score dtype: {c['bat_score'].dtype}")
print(f"bat_score unique values (first 20): {sorted(c['bat_score'].dropna().unique())[:20]}")

## 3. 2.1.1 -- Cap comments per post

Sysadmin threads average 27 top-level comments; without a cap, a few large threads would define the concept set.

In [ ]:
capped = (c.groupby("post_id", group_keys=False)
           .apply(lambda t: t.sample(min(len(t), 3), random_state=42)))
print(f"{len(c):,} comments -> {len(capped):,} after capping at 3/post")

## 4. 2.1.2 -- Severity band

Check the `bat_score` dtype/values printed in step 2 first -- this only produces clean bands ("1","2","3") if `bat_score` is a small integer count. If it's continuous, `.astype(str)` after `.clip()` will explode into near-unique values instead of real bins, and `sev_band.nunique()` in the next step will be far larger than expected.

In [ ]:
capped["sev_band"] = capped.bat_score.clip(upper=3).astype(str)   # "0" won't appear

print("sev_band value counts:")
print(capped["sev_band"].value_counts().sort_index())

n_bands = capped["sev_band"].nunique()
if n_bands > 10:
    print(f"\nWARNING: {n_bands} distinct severity bands found -- expected ~3-4. "
          f"bat_score is likely continuous; consider .round(0) before .astype(str), "
          f"or a proper pd.cut() binning, before proceeding.")
else:
    print(f"\n{n_bands} severity bands -- looks reasonable, proceeding.")

## 5. 2.1.3 -- Draw with fixed per-subreddit quotas

Using your exact `GEN_N` values (sums to 2,176, not exactly 2,200 -- kept as specified since you said "about 2,200").

In [ ]:
GEN_N = {"sysadmin": 700, "cybersecurity": 500,
         "SecurityCareerAdvice": 400, "asknetsec": 400, "ciso": 176}

print(f"GEN_N total: {sum(GEN_N.values())}")

missing = set(GEN_N.keys()) - set(capped[SUBREDDIT_COL].unique())
if missing:
    raise ValueError(
        f"GEN_N references subreddit(s) not found in the data: {missing}\n"
        f"Actual values are: {sorted(capped[SUBREDDIT_COL].unique())}"
    )
def draw(g, n):
    per = max(1, n // g["sev_band"].nunique())

    def sample_band(b):
        sampled = b.sample(min(len(b), per), random_state=42).copy()
        sampled["sev_band"] = b.name   # re-attach -- newer pandas strips the groupby column from b
        return sampled

    out = g.groupby("sev_band", group_keys=False).apply(sample_band)

    if len(out) < n:
        rest = g.drop(out.index, errors="ignore")
        out = pd.concat([out, rest.sample(min(len(rest), n - len(out)), random_state=42)])
    return out


def draw_wrapper(g):
    sub_out = draw(g, min(GEN_N[g.name], len(g))).copy()
    sub_out[SUBREDDIT_COL] = g.name   # re-attach -- same issue, one level up
    return sub_out

gen_sample = capped.groupby(SUBREDDIT_COL, group_keys=False).apply(draw_wrapper)
print(f"\nGeneration sample: {len(gen_sample):,} comments")
print(gen_sample.groupby([SUBREDDIT_COL, "sev_band"]).size())

In [ ]:
%pip install pyarrow -q

In [ ]:
GEN_SAMPLE_PATH = os.path.join(OUTPUT_DIR, "gen_sample.parquet")
gen_sample.to_parquet(GEN_SAMPLE_PATH)
print(f"Saved: {GEN_SAMPLE_PATH}")
print("Every later step must use this exact sample -- reload from this file, don't re-draw.")

## 6. Model setup (Gemini 3.7 Flash via OpenRouter, same validated wiring)

In [ ]:
import getpass

OPENROUTER_API_KEY = getpass.getpass("OpenRouter API key: ")
OPENROUTER_MODEL = "openai/gpt-5.6-luna-pro"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

GEMINI_COST = (0.375 / 1_000_000, 1.875 / 1_000_000)  # verify current pricing
CONTEXT_WINDOW = 1_048_576
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"


In [ ]:
def setup_llm_fn(api_key):
    from openai import AsyncOpenAI
    import httpx
    # Default timeout is connect=5.0s, read/write/pool=600s -- 5s is too
    # aggressive under heavy congestion (causes APITimeoutError independent
    # of 429s). Widened connect to 15s, brought others down from 600s since
    # these are short scoring calls, not long generations.
    return AsyncOpenAI(
        api_key=api_key,
        base_url=OPENROUTER_BASE_URL,
        timeout=httpx.Timeout(connect=15.0, read=120.0, write=120.0, pool=120.0),
    )

def setup_embed_fn(api_key):
    from fastembed import TextEmbedding
    return TextEmbedding(model_name=EMBED_MODEL_NAME)

import asyncio
import random

MAX_RETRIES = 5
BASE_DELAY = 5

# Raised from 2048 -> 4096: batched scoring responses (multiple examples'
# rationale+quote+answer per call) were getting truncated mid-JSON at 2048.
MAX_OUTPUT_TOKENS = 4096

async def call_llm_fn(model, prompt):
    if "system_prompt" not in model.args:
        model.args["system_prompt"] = (
            "You are a helpful assistant who helps with identifying patterns "
            "in text examples."
        )
    if "temperature" not in model.args:
        model.args["temperature"] = 0

    for attempt in range(MAX_RETRIES):
        try:
            res = await model.client.chat.completions.create(
                model=model.name,
                temperature=model.args["temperature"],
                max_tokens=MAX_OUTPUT_TOKENS,
                messages=[
                    {"role": "system", "content": model.args["system_prompt"]},
                    {"role": "user", "content": prompt},
                ],
            )
            text = res.choices[0].message.content if res and res.choices else None
            if res and getattr(res, "usage", None):
                tokens = (res.usage.prompt_tokens, res.usage.completion_tokens)
            else:
                tokens = (0, 0)
            return text, tokens

        except Exception as e:
            err_str = str(e).lower()
            is_credits = "402" in str(e) or "requires more credits" in err_str
            is_last_attempt = attempt == MAX_RETRIES - 1

            is_retryable = (

                "429" in str(e)
                or "rate limit" in err_str
                or "timed out" in err_str
                or "timeout" in err_str
                or "connection" in err_str
            )

            if is_credits:
                print(f"  [402 -- out of credits] {e}")
                print("  Add credits at https://openrouter.ai/settings/credits")
                return None, None

            if is_retryable and not is_last_attempt:
                delay = BASE_DELAY * (2 ** attempt) + random.uniform(0, 2)
                print(f"  [{type(e).__name__}] retrying in {delay:.1f}s (attempt {attempt + 1}/{MAX_RETRIES})...")
                await asyncio.sleep(delay)
                continue
            else:
                print(f"  [error, giving up after {attempt + 1} attempt(s)]: {e}")
                return None, None

    return None, None

def call_embed_fn(model, text_arr):
    embeddings = list(model.client.embed(text_arr))
    embeddings = [e.tolist() for e in embeddings]
    tokens = (0, 0)
    return embeddings, tokens

In [ ]:
def build_models():
    """Fresh Model/EmbedModel instances -- also used to reattach models to a
    session reloaded from pickle, since l.save() nulls them out before writing
    to disk (they can't be pickled)."""
    return dict(
        distill_model=Model(
            setup_fn=setup_llm_fn, fn=call_llm_fn, name=OPENROUTER_MODEL,
            cost=GEMINI_COST, rate_limit=(15, 10), context_window=CONTEXT_WINDOW, api_key=OPENROUTER_API_KEY,
        ),
        cluster_model=EmbedModel(
            setup_fn=setup_embed_fn, fn=call_embed_fn, name=EMBED_MODEL_NAME,
            cost=0, batch_size=64, api_key=OPENROUTER_API_KEY,
        ),
        synth_model=Model(
            setup_fn=setup_llm_fn, fn=call_llm_fn, name=OPENROUTER_MODEL,
            cost=GEMINI_COST, rate_limit=(10, 10), context_window=CONTEXT_WINDOW, api_key=OPENROUTER_API_KEY,
        ),
        score_model=Model(
            setup_fn=setup_llm_fn, fn=call_llm_fn, name=OPENROUTER_MODEL,
            cost=GEMINI_COST, rate_limit=(10, 10), context_window=CONTEXT_WINDOW, api_key=OPENROUTER_API_KEY,
        ),
    )

In [ ]:
%pip install python-dotenv -q

from dotenv import load_dotenv
import os

# Point directly at your .env file -- don't rely on load_dotenv()'s automatic
# upward search, since the notebook's working directory doesn't always match
# wherever the file actually sits, and a mismatch fails silently (returns
# False) rather than raising an error.
loaded = load_dotenv("/Users/nadia/Desktop/redditRun_june/comment_data/.env")  # ⚠️ EDIT ME to the real path
print(f".env loaded: {loaded}")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

print(f"Key set: {bool(OPENROUTER_API_KEY)}, length: {len(OPENROUTER_API_KEY) if OPENROUTER_API_KEY else 0}")
if OPENROUTER_API_KEY:
    print(f"Starts with: {OPENROUTER_API_KEY[:8]}...")

## 7. 2.2 -- Generate the real concepts (unseeded, then seeded)

Two separate `lloom` instances (`l_unseeded`, `l_seeded`) rather than reusing one -- keeps the two runs' state (`df_filtered`, `df_bullets`, etc.) unambiguous, even though `gen()` does reset `self.concepts` internally between calls.

**`debug=True` is required** for `auto_review` to actually run -- the library nests the Review step inside `if debug:`. This means you'll get an interactive `Proceed with generation? (y/n)` prompt for each of the two calls below; type `y` directly into the cell's input prompt when it appears.

Checkpointed: re-running this cell skips whichever of the two runs already has a saved checkpoint.

In [ ]:
CHOSEN_SEED = "what the commenter is doing in relation to the person who wrote the post"  # Seed A, from Phase 1

CKPT_DIR = os.path.join(OUTPUT_DIR, "ckpt")
os.makedirs(CKPT_DIR, exist_ok=True)

gen_sample = pd.read_parquet(GEN_SAMPLE_PATH)  # reload from the frozen file, not the in-memory var

# --- Unseeded run ---
unseeded_path = os.path.join(CKPT_DIR, "final_unseeded.pkl")
if os.path.exists(unseeded_path):
    print("[unseeded] checkpoint found -- loading instead of re-running gen()")
    with open(unseeded_path, "rb") as f:
        l_unseeded = pickle.load(f)
    models = build_models()
    l_unseeded.distill_model = models["distill_model"]
    l_unseeded.cluster_model = models["cluster_model"]
    l_unseeded.synth_model = models["synth_model"]
    l_unseeded.score_model = models["score_model"]
else:
    print("[unseeded] running gen()...")
    l_unseeded = wb.lloom(df=gen_sample, text_col=TEXT_COL, id_col=ID_COL, **build_models())
    params = l_unseeded.auto_suggest_parameters(target_n_concepts=20)
    print(f"Auto-suggested params: {params}")
    await l_unseeded.gen(params=params, n_synth=2, auto_review=True, debug=True)
    l_unseeded.save(folder=CKPT_DIR, file_name="final_unseeded")

# --- Seeded run ---
seeded_path = os.path.join(CKPT_DIR, "final_seeded.pkl")
if os.path.exists(seeded_path):
    print("\n[seeded] checkpoint found -- loading instead of re-running gen()")
    with open(seeded_path, "rb") as f:
        l_seeded = pickle.load(f)
    models = build_models()
    l_seeded.distill_model = models["distill_model"]
    l_seeded.cluster_model = models["cluster_model"]
    l_seeded.synth_model = models["synth_model"]
    l_seeded.score_model = models["score_model"]
else:
    print("\n[seeded] running gen()...")
    l_seeded = wb.lloom(df=gen_sample, text_col=TEXT_COL, id_col=ID_COL, **build_models())
    params = l_seeded.auto_suggest_parameters(target_n_concepts=20)
    print(f"Auto-suggested params: {params}")
    await l_seeded.gen(params=params, seed=CHOSEN_SEED, n_synth=2, auto_review=True, debug=True)
    l_seeded.save(folder=CKPT_DIR, file_name="final_seeded")

print(f"\nUnseeded: {len(l_unseeded.concepts)} concepts")
print(f"Seeded: {len(l_seeded.concepts)} concepts")

[unseeded] running gen()...
Auto-suggested params: {'filter_n_quotes': 3, 'summ_n_bullets': 2, 'synth_n_concepts': 6}
Cost estimates not available for distill model `openai/gpt-5.6-luna-pro`
Cost estimates not available for cluster model `BAAI/bge-small-en-v1.5`
Cost estimates not available for synth model `openai/gpt-5.6-luna-pro`


Action required


Distill-filter
⠦ Loading   [error, giving up after 1 attempt(s)]: Error code: 401 - {'error': {'message': 'No cookie auth credentials found', 'code': 401}}
  [error, giving up after 1 attempt(s)]: Error code: 401 - {'error': {'message': 'No cookie auth credentials found', 'code': 401}}
  [error, giving up after 1 attempt(s)]: Error code: 401 - {'error': {'message': 'No cookie auth credentials found', 'code': 401}}
  [error, giving up after 1 attempt(s)]: Error code: 401 - {'error': {'message': 'No cookie auth credentials found', 'code': 401}}
  [error, giving up after 1 attempt(s)]: Error code: 401 - {'error': {'message': 'No cookie auth 

CancelledError: 

## 8. Print both concept lists

In [ ]:
print("===== UNSEEDED =====")
print(f"(n={len(l_unseeded.concepts)}):")
for cpt in l_unseeded.concepts.values():
    print(f"- {cpt.name}: {cpt.prompt}")

print("\n===== SEEDED =====")
print(f"(n={len(l_seeded.concepts)}):")
for cpt in l_seeded.concepts.values():
    print(f"- {cpt.name}: {cpt.prompt}")

===== UNSEEDED =====
(n=9):
- Online Community Moderation: Determine whether the text discusses forum moderation actions, rule enforcement, post removals, or thread redirections.
- Job Search Strategies: Determine whether the text provides guidance, tactics, or advice for navigating the job hunting process, such as preparing applications, interviewing, or managing a competitive market.
- Strategic Career Transitions: Determine whether the text discusses leaving a current job or switching employers to escape toxic environments or secure higher compensation.
- Professional Skill Development: Determine whether the text emphasizes continuous learning, technical training, or skill acquisition to enhance career growth and performance.
- Cybersecurity Career Development: Determine whether the text provides advice, prerequisites, strategies, or certification guidance for entering or advancing in IT and cybersecurity careers.
- Security Operations and Threats: Determine whether the text focuses

---
## Section 2.3 support code (run interactively, not top-to-bottom)

The cells below are the tools from your plan, ready to use once you've decided how to reconcile the unseeded/seeded lists into one working set (assign it to `l` below). This section is iterative -- run 2.3.1, read the review sheets, edit criteria, rescore, repeat.

In [ ]:
# Pick which lloom instance (or a manually reconciled concept set) to review.
# Defaulting to l_seeded since that's the validated seed from Phase 1 -- change
# if you're reconciling both lists into a combined set instead.

l = l_seeded
l.select()

### 2.3.1 -- Score the generation sample

### Diagnostic -- run this BEFORE the score() cell below

Checks two separate things that both produce the same "0it" / empty-result symptom, so we stop guessing which one it is:
1. **Row content**: does `gen_sample`'s text column actually have real text, or are rows empty/null (would get silently dropped by LLooM's internal `filter_empty_rows`)?
2. **Concept state**: does `l.concepts` actually have entries, and are any marked `active=True` (this is what `l.select()` is supposed to set, but it's a non-blocking widget -- easy for this to end up empty if selections weren't made before this cell runs)?

In [ ]:
# --- 1. Row content check ---
n_total = len(gen_sample)
n_null = gen_sample[TEXT_COL].isna().sum()
n_empty_str = (gen_sample[TEXT_COL].astype(str).str.strip() == "").sum()
print(f"gen_sample: {n_total:,} rows")
print(f"  null {TEXT_COL}: {n_null:,}")
print(f"  empty-string {TEXT_COL}: {n_empty_str:,}")
if n_null + n_empty_str == n_total:
    print("  ⚠️  ALL rows are empty/null -- this would explain zero tasks downstream.")
elif n_null + n_empty_str > 0:
    print(f"  Some empty rows present, but {n_total - n_null - n_empty_str:,} have real text -- "
          f"not enough to zero out the whole task list on its own.")
else:
    print("  ✅ No empty/null rows -- row content is not the cause.")

print(f"\nSample text (first 3 non-null rows):")
for t in gen_sample[TEXT_COL].dropna().head(3):
    print(f"  - {t[:150]!r}")

# --- 2. Concept active-state check ---
print(f"\nTotal concepts on l: {len(l.concepts)}")
n_active = sum(1 for c in l.concepts.values() if c.active)
print(f"Active concepts: {n_active}")
for c_id, c in l.concepts.items():
    print(f"  {c_id}: active={c.active}  name={c.name}")

if len(l.concepts) == 0:
    print("\n⚠️  l.concepts is completely empty -- something went wrong in generation "
          "or the checkpoint reload, not in select()/score().")
elif n_active == 0:
    print("\n⚠️  Concepts exist but NONE are active -- l.select() ran but no selections "
          "were registered before this cell ran. Go back, run l.select() again, actually "
          "click checkboxes in the widget, THEN come back to this diagnostic and the score() "
          "cell below.")
else:
    print(f"\n✅ {n_active} active concept(s) -- score() should have something to work with.")

In [ ]:
# ignore_existing=False: without this, l.score() silently skips any concept
# that already has an entry in l.results -- including from an earlier
# interrupted/failed attempt -- which can zero out the task list the same
# way an empty concepts list does.
# df=gen_sample: explicit, rather than relying on l's internal df_to_score
# cache, which may not match this exact sample if regenerated.
sc = await l.score(get_highlights=True, debug=True, batch_size=5, ignore_existing=False, df=gen_sample)

Cost estimates not available for score model `google/gemini-3.7-flash`


Action required
Proceed with scoring? (y/n): y
0it [00:00, ?it/s]


ValueError: No objects to concatenate

In [ ]:
sc.shape if 'sc' in globals() else "not finished yet"

'not finished yet'

In [ ]:
sc.to_parquet(os.path.join(OUTPUT_DIR, "gen_scores.parquet"))
print(f"Scored {len(sc):,} rows")

### 2.3.2 -- Review sheet per concept

In [ ]:
for cid, grp in sc[sc.score > 0].groupby("concept_id"):
    name = grp.concept_name.iloc[0]
    prompt = grp.concept_prompt.iloc[0]
    n = len(grp)
    print(f"\n{'='*70}\nCONCEPT {cid}: {name}")
    print(f"CRITERIA: {prompt}")
    print(f"PREVALENCE: {n} / {gen_sample.shape[0]} = {n/gen_sample.shape[0]:.1%}\n")
    for t in grp.text.sample(min(20, n), random_state=0):
        print(f"  - {t[:250]}")

### 2.3.3 -- Four questions per concept (spreadsheet, done by hand)

| Question | If no |
|---|---|
| Do the 20 examples share the thing the name claims? | Rewrite criteria, or drop |
| Is the name an action rather than a topic? | Rewrite name and criteria |
| Is it distinct from every other concept? | Merge |
| Is prevalence between ~2% and 50%? | Under 2%: drop as too rare. Over 50%: split or drop as non-discriminating |


### 2.3.4 -- Mechanical redundancy check (Jaccard)

In [ ]:
w = sc.pivot(index="doc_id", columns="concept_name", values="score").fillna(0)
for a, b in itertools.combinations(w.columns, 2):
    inter = ((w[a] == 1) & (w[b] == 1)).sum()
    union = ((w[a] == 1) | (w[b] == 1)).sum()
    j = inter / union if union else 0
    if j > 0.5:
        print(f"MERGE CANDIDATE  {j:.2f}  {a}  ||  {b}")

### 2.3.5 -- Rewrite and rescore

After editing criteria (directly on `l.concepts[c_id].prompt`), re-run 2.3.1 through 2.3.4. Usually one or two rounds.

### 2.3.6 -- Freeze to JSON (target 12-16 concepts)

In [ ]:
# Build `final_concepts` as a list of {"name":..., "prompt":...} dicts once
# you've settled on the final set -- e.g.:
# final_concepts = [{"name": c.name, "prompt": c.prompt} for c in l.concepts.values() if c.active]

frozen = [{"id": f"L{i:02d}", "source": "lloom", "name": r["name"], "prompt": r["prompt"]}
          for i, r in enumerate(final_concepts)]
json.dump(frozen, open(os.path.join(OUTPUT_DIR, "frozen_concepts.json"), "w"), indent=2)
print(f"Froze {len(frozen)} concepts. Nothing after this point should change this file.")